In [9]:
!pip install stanza scikit-learn

import pandas as pd
import stanza

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [10]:
from google.colab import drive
drive.mount('/content/drive')
# download processed file to your drive

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
df_text = pd.read_csv('drive/MyDrive/Colab Notebooks/NLP/processed_v2.xls')
df_labels = pd.read_csv('drive/MyDrive/Colab Notebooks/NLP/labels.xls')

# assume labels file has single column
df_text["label"] = df_labels.iloc[:, 0]

df = df_text.dropna(subset=["statement_1", "statement_2", "label"])


df["pair_text"] = (
    df["statement_1"].astype(str) + " [SEP] " +
    df["statement_2"].astype(str)
)

In [14]:
stanza.download("uk")
nlp = stanza.Pipeline("uk", processors="tokenize,pos,lemma", use_gpu=False)

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: uk (Ukrainian) ...


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/uk/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: uk (Ukrainian):
| Processor | Package     |
---------------------------
| tokenize  | iu          |
| mwt       | iu          |
| pos       | iu_charlm   |
| lemma     | iu_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Done loading processors!


In [15]:
def lemmatize(text):
    doc = nlp(text)
    lemmas = []
    pos_tags = []

    for sent in doc.sentences:
        for word in sent.words:
            lemmas.append(word.lemma.lower())
            pos_tags.append(word.upos)

    return " ".join(lemmas), " ".join(pos_tags)

df[["lemma_text", "upos_seq"]] = df["pair_text"].apply(
    lambda x: pd.Series(lemmatize(x))
)

In [16]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["pair_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

X_train_lemma, X_test_lemma, _, _ = train_test_split(
    df["lemma_text"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [17]:
vectorizer_raw = TfidfVectorizer(ngram_range=(1,2), max_features=20000)

Xtr_raw = vectorizer_raw.fit_transform(X_train_raw)
Xte_raw = vectorizer_raw.transform(X_test_raw)

model_raw = LogisticRegression(max_iter=1000)
model_raw.fit(Xtr_raw, y_train)

pred_raw = model_raw.predict(Xte_raw)

print("Baseline 1 — raw")
print("Accuracy:", accuracy_score(y_test, pred_raw))
print("Macro-F1:", f1_score(y_test, pred_raw, average="macro"))

Baseline 1 — raw
Accuracy: 0.636
Macro-F1: 0.40388848129126925


In [18]:
vectorizer_lemma = TfidfVectorizer(ngram_range=(1,2), max_features=20000)

Xtr_lemma = vectorizer_lemma.fit_transform(X_train_lemma)
Xte_lemma = vectorizer_lemma.transform(X_test_lemma)

model_lemma = LogisticRegression(max_iter=1000)
model_lemma.fit(Xtr_lemma, y_train)

pred_lemma = model_lemma.predict(Xte_lemma)

print("\nBaseline 2 — lemma")
print("Accuracy:", accuracy_score(y_test, pred_lemma))
print("Macro-F1:", f1_score(y_test, pred_lemma, average="macro"))


Baseline 2 — lemma
Accuracy: 0.634
Macro-F1: 0.3981490617045208


In [19]:
errors_idx = X_test_raw.index[pred_raw != y_test]

for i in errors_idx[:10]:
    print("="*50)
    print("Statement 1:", df.loc[i, "statement_1"])
    print("Statement 2:", df.loc[i, "statement_2"])
    print("True:", y_test.loc[i])
    print("Pred:", pred_raw[list(X_test_raw.index).index(i)])

    doc = nlp(df.loc[i, "pair_text"])

    for sent in doc.sentences:
        for word in sent.words:
            print(f"{word.text} | lemma: {word.lemma} | POS: {word.upos}")

    print("Comment: manual analysis")

Statement 1: Привіт іще раз.
Statement 2: Таксі подано.
True: 1
Pred: 0
Привіт | lemma: привіт | POS: NOUN
іще | lemma: іще | POS: ADV
раз | lemma: раз | POS: NOUN
. | lemma: . | POS: PUNCT
[SEP | lemma: [SEP | POS: SYM
] | lemma: ] | POS: PUNCT
Таксі | lemma: таксі | POS: NOUN
подано | lemma: подати | POS: VERB
. | lemma: . | POS: PUNCT
Comment: manual analysis
Statement 1: У Тома дивне почуття гумору.
Statement 2: Том має дивне почуття гумору.
True: 1
Pred: 0
У | lemma: у | POS: ADP
Тома | lemma: Том | POS: PROPN
дивне | lemma: дивний | POS: ADJ
почуття | lemma: почуття | POS: NOUN
гумору | lemma: гумора | POS: NOUN
. | lemma: . | POS: PUNCT
[SEP | lemma: [SEP | POS: SYM
] | lemma: ] | POS: PUNCT
Том | lemma: том | POS: NOUN
має | lemma: мати | POS: VERB
дивне | lemma: дивний | POS: ADJ
почуття | lemma: почуття | POS: NOUN
гумору | lemma: гумора | POS: NOUN
. | lemma: . | POS: PUNCT
Comment: manual analysis
Statement 1: Яка їжа є у вас?
Statement 2: Що в вас є поїсти?
True: 1
Pred: 0

У нашому експерименті використання lemma_text не дало суттєвого покращення baseline, а в частині випадків якість навіть знизилась. Зокрема, модель не змогла правильно визначити семантичну еквівалентність у парафразах типу «У мене є кіт» — «Я маю кота» або «Чому ти це зробив?» — «Нащо ти це зробив?», хоча леми формально збігалися частково. Також спостерігаються помилки лематизації та POS-тегування власних назв і окремих форм (наприклад, «Том» як NOUN замість PROPN), що потенційно впливає на якість ознак. Лематизація не допомогла у випадках синонімічних конструкцій («Бога немає» — «Бога не існує», «Я не заперечую» — «Я не проти»), оскільки TF-IDF не враховує глибинну семантику. Для нашого проєкту доцільно не використовувати леми як основне представлення для класифікації